# Machine-Learning Preprocessing

This notebook prepares the feature-engineered building data for hierarchical machine-learning classification. It defines the predictor set, constructs the training and prediction populations, encodes target classes, creates reproducible train/validation/test splits, and generates model-specific datasets for each level of the classification hierarchy.

**Input**
- Feature-engineered building data with Stage 1 and Stage 2 classifications.

**Output**
- Train, validation, and test datasets for each hierarchical classification task.
- Prediction datasets for unresolved buildings and buildings requiring subtype classification.
- Encoders and preprocessing objects required for model training and inference.

In [23]:
import pandas as pd
import geopandas as gpd
import numpy as np
import joblib
import os
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
warnings.filterwarnings('ignore')

ROOT_DIR = '/fast/home/o-olajuyigbe/osm_project'
DATA_DIR = os.path.join(ROOT_DIR, 'data')
PROC_DIR  = os.path.join(DATA_DIR, 'processed')
ML_DIR    = os.path.join(DATA_DIR, 'ml_ready')
os.makedirs(ML_DIR, exist_ok=True)


BUILDING_FILE = os.path.join(PROC_DIR, 'germany_buildings_feature_engineered.parquet')
OUTPUT_FILE = os.path.join(ML_DIR, 'germany_buildings_ml_ready.parquet')



In [24]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [25]:
print("Loading Data...")

# Load Parquet files
gdf_bldg = gpd.read_parquet(BUILDING_FILE)

print("Data loaded successfully!")

print(f'Shape: {gdf_bldg.shape}')
print(f'Columns ({len(gdf_bldg.columns)}):\n', list(gdf_bldg.columns))

Loading Data...
Data loaded successfully!
Shape: (38802372, 156)
Columns (156):
 ['id', 'geometry', 'building', 'tag_l1', 'tag_l2', 'is_abandoned', 'tag_is_mixed', 'tag_source', 'tag_used', 'all_candidates', 'addr:city', 'addr:housenumber', 'addr:postcode', 'addr:street', 'name', 'amenity', 'building:use', 'craft', 'office', 'shop', 'tags', 'address_tier', 'compactness', 'building_l1', 'building_l2', 'building_is_mixed', 'buse_l1', 'buse_l2', 'buse_is_mixed', 'amenity_l1', 'amenity_l2', 'shop_l1', 'shop_l2', 'stage1_l1', 'stage1_l2', 'stage1_is_mixed', 'stage1_source', 'raw_label', 'stage2_l1', 'stage2_l2', 'stage2_source', 'landuse_l1', 'landuse_l2', 'landuse_used', 'landuse_osm_id', 'has_building_levels', 'building_levels', 'has_name', 'area', 'perimeter', 'convexity', 'num_vertices', 'mrr_long', 'mrr_short', 'elongation', 'rectangularity', 'diameter', 'dist_nearest_real_building', 'is_touching_real_building', 'real_touching_count', 'dist_to_residential', 'dist_to_commercial', 'dist_

In [26]:
# set id as index
gdf_bldg = gdf_bldg.set_index('id')

print(f"Index set to 'id'. Shape: {gdf_bldg.shape}")


Index set to 'id'. Shape: (38802372, 155)


### Column Groups

In [27]:
DROP_IDENTIFIERS = ['geometry',
    'addr:city', 'addr:housenumber', 'addr:postcode', 'addr:street',
    'name', 'tags',
]

DROP_RAW_TAGS = [
    'building', 'amenity', 'building:use', 'craft', 'office', 'shop',
]

DROP_PIPELINE = [
    'stage1_source', 'stage1_is_mixed',
    'stage2_source', 'is_abandoned',
    'landuse_used', 'landuse_osm_id',
    'buse_is_mixed', 'building_is_mixed',
    'tag_source', 'tag_used', 'tag_is_mixed',
    'all_candidates', 'raw_label',
]

DROP_LABEL_LEAKAGE = [  
    'building_l1', 'building_l2',
    'buse_l1', 'buse_l2',
    'amenity_l1', 'amenity_l2',
    'shop_l1', 'shop_l2',
    'landuse_l1', 'landuse_l2',  
    'tag_l1', 'tag_l2',
]

TARGETS = ['stage1_l1', 'stage1_l2', 'stage2_l1', 'stage2_l2',] 

FEATURE_DROP = ['has_building_levels', 'has_name', 'building_levels']

ALL_DROP = (
    DROP_IDENTIFIERS + DROP_RAW_TAGS +
    DROP_PIPELINE + DROP_LABEL_LEAKAGE + FEATURE_DROP
)

# Only drop columns that actually exist
ALL_DROP = [c for c in ALL_DROP if c in gdf_bldg.columns]
gdf_bldg.drop(columns=ALL_DROP, inplace=True)

# Feature columns = everything except targets
feature_cols = [c for c in gdf_bldg.columns if c not in TARGETS]

print(f"Shape after drops: {gdf_bldg.shape}")
print(f"Feature columns ({len(feature_cols)}):")
for c in feature_cols:
    print(f"  {c}  [{gdf_bldg[c].dtype}]")

Shape after drops: (38802372, 114)
Feature columns (110):
  address_tier  [int64]
  compactness  [float64]
  area  [float64]
  perimeter  [float64]
  convexity  [float64]
  num_vertices  [int64]
  mrr_long  [float64]
  mrr_short  [float64]
  elongation  [float64]
  rectangularity  [float64]
  diameter  [float64]
  dist_nearest_real_building  [float32]
  is_touching_real_building  [int8]
  real_touching_count  [int8]
  dist_to_residential  [float32]
  dist_to_commercial  [float32]
  dist_to_industrial  [float32]
  dist_to_agricultural  [float32]
  buildings_in_same_zone  [int32]
  zone_total_area  [float32]
  building_area_fraction  [float32]
  zone_building_density  [float32]
  neighbour_mean_area_200m  [float32]
  neighbour_max_area_200m  [float32]
  neighbour_std_area_200m  [float32]
  poi_count_retail_25m  [int32]
  poi_count_retail_50m  [int32]
  poi_count_retail_100m  [int32]
  poi_count_retail_250m  [int32]
  poi_count_office_25m  [int32]
  poi_count_office_50m  [int32]
  poi_cou

In [28]:
# Quick null audit for features
null_pct = (gdf_bldg[feature_cols].isnull().sum() / len(gdf_bldg) * 100).sort_values(ascending=False)
print('Null % per column (top 5):')
print(null_pct.head(5).round(2))

Null % per column (top 5):
address_tier    0.0
compactness     0.0
area            0.0
perimeter       0.0
convexity       0.0
dtype: float64


In [29]:
# float64 → float32 for the geometry features 
float64_cols = [
    'compactness', 'area', 'perimeter', 'convexity',
    'mrr_long', 'mrr_short', 'elongation', 'rectangularity',
    'diameter'
]
gdf_bldg[float64_cols] = gdf_bldg[float64_cols].astype(np.float32)

# int64 → int32 
gdf_bldg['num_vertices']   = gdf_bldg['num_vertices'].astype(np.int16)  

In [30]:
gdf_bldg[feature_cols].info()

<class 'pandas.core.frame.DataFrame'>
Index: 38802372 entries, 3428357 to 1490361947
Columns: 110 entries, address_tier to count_heavy_rail_200m
dtypes: float32(51), int16(12), int32(34), int64(1), int8(12)
memory usage: 14.2+ GB


In [31]:
# Split Into Three Sets

# TRAINING SET: has a stage 1 label AND it is NOT filter AND it is NOT semi_commercial
# also exclude where stage1_l2 = 'other_service' 
df_train_pool = gdf_bldg[
    gdf_bldg['stage1_l1'].notna() &
    (~gdf_bldg['stage1_l1'].isin(['filter', 'semi_commercial'])) &
    (~gdf_bldg['stage1_l2'].isin(['other_service']))
].copy()

# PREDICT SET: ML needs to classify these (both stages are empty)
df_predict = gdf_bldg[
    gdf_bldg['stage1_l1'].isna() &
    (gdf_bldg['stage2_l1'].isna())
].copy()

# STAGE2-ONLY SET: no stage1 label but has stage2 label
# Buildings classified by rule-based classification
df_stage2 = gdf_bldg[
    gdf_bldg['stage1_l1'].isna() &
    gdf_bldg['stage2_l1'].notna() 
].copy()

# EXCLUDED SET: filter rows where stage1 = filter or stage1 = semi_commercial 
# also exclude where stage1_l2 = 'other_service'
df_excluded = gdf_bldg[
    gdf_bldg['stage1_l1'].isin(['filter', 'semi_commercial']) |
    gdf_bldg['stage1_l2'].isin(['other_service'])
].copy()

# Drop targets from predict 
X_predict = df_predict[feature_cols].copy()

print(f"Training pool  : {len(df_train_pool):,}")
print(f"Predict set    : {len(df_predict):,}")
print(f"Stage2-only set: {len(df_stage2):,}")
print(f"Excluded set   : {len(df_excluded):,}")
print(f"Total check    : {len(df_train_pool) + len(df_predict) + len(df_stage2) + len(df_excluded):,}")

print(f"\nTraining label distribution (stage1_l1):")
print(df_train_pool['stage1_l1'].value_counts())
print(df_train_pool['stage1_l1'].value_counts(normalize=True).round(3))


Training pool  : 9,143,593
Predict set    : 18,259,833
Stage2-only set: 7,270,447
Excluded set   : 4,128,499
Total check    : 38,802,372

Training label distribution (stage1_l1):
stage1_l1
residential       7765260
commercial         420912
agricultural       313609
civic              311495
industrial         301728
transportation      22301
military             8288
Name: count, dtype: int64
stage1_l1
residential       0.849
commercial        0.046
agricultural      0.034
civic             0.034
industrial        0.033
transportation    0.002
military          0.001
Name: proportion, dtype: float64


In [32]:
print(X_predict.shape)
X_predict.head()


(18259833, 110)


,address_tier,compactness,area,perimeter,convexity,num_vertices,mrr_long,mrr_short,elongation,rectangularity,diameter,dist_nearest_real_building,is_touching_real_building,real_touching_count,dist_to_residential,dist_to_commercial,dist_to_industrial,dist_to_agricultural,buildings_in_same_zone,zone_total_area,building_area_fraction,zone_building_density,neighbour_mean_area_200m,neighbour_max_area_200m,neighbour_std_area_200m,poi_count_retail_25m,poi_count_retail_50m,poi_count_retail_100m,poi_count_retail_250m,poi_count_office_25m,poi_count_office_50m,poi_count_office_100m,poi_count_office_250m,poi_count_food_25m,poi_count_food_50m,poi_count_food_100m,poi_count_food_250m,poi_count_other_service_25m,poi_count_other_service_50m,poi_count_other_service_100m,poi_count_other_service_250m,poi_count_accommodation_25m,poi_count_accommodation_50m,poi_count_accommodation_100m,poi_count_accommodation_250m,poi_count_healthcare_25m,poi_count_healthcare_50m,poi_count_healthcare_100m,poi_count_healthcare_250m,poi_count_education_25m,poi_count_education_50m,poi_count_education_100m,poi_count_education_250m,dist_nearest_education,poi_count_civic_25m,poi_count_civic_50m,poi_count_civic_100m,poi_count_civic_250m,dist_nearest_transport,food_share_100m,retail_share_100m,office_share_100m,accom_share_100m,food_vs_retail_100m,office_vs_retail_100m,comm_diversity_100m,food_share_250m,retail_share_250m,office_share_250m,accom_share_250m,food_vs_retail_250m,office_vs_retail_250m,comm_diversity_250m,poi_count_inside,dist_nearest_residential_road,dist_nearest_secondary_road,dist_nearest_rural_track,dist_nearest_major_road,dist_nearest_service_road,dist_nearest_pedestrian_zone,near_major_road_700m,near_secondary_road_115m,near_residential_road_25m,near_service_road_35m,near_rural_track_250m,near_pedestrian_zone_25m,road_count_residential_road_200m,road_count_secondary_road_200m,road_count_rural_track_200m,road_count_major_road_200m,road_count_service_road_200m,road_count_pedestrian_zone_200m,road_count_total_200m,pct_residential_roads_200m,pct_rural_roads_200m,pct_service_roads_200m,pct_major_roads_200m,pct_secondary_roads_200m,dist_nearest_major_waterway,near_major_waterway_500m,count_major_waterway_200m,dist_nearest_minor_waterway,near_minor_waterway_300m,count_minor_waterway_200m,dist_nearest_light_rail,near_light_rail_200m,count_light_rail_200m,dist_nearest_heavy_rail,near_heavy_rail_500m,count_heavy_rail_200m
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3428357,2,0.682492,370.210480,82.562065,0.984236,12,27.623497,14.003118,1.972668,0.957074,30.970064,30.674368,0,0,427.137604,620.840149,540.566223,599.406372,0,0.000000,0.000000,0.000000,150.816696,1280.692017,188.410172,0,0,0,42,0,0,0,2,0,0,0,26,0,0,0,10,0,0,0,0,0,0,0,4,0,0,0,0,275.963715,0,0,0,6,2532.412354,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.324996,0.524993,0.025,0.0,0.382347,0.045454,0.618135,0,45.459679,295.174438,339.831268,1137.595947,17.225588,2387.968506,0,0,0,1,0,0,62,0,0,0,20,0,82,0.756098,0.0,0.243902,0.0,0.0,774.688782,0,0,243.551819,1,0,5000.0,0,0,392.610382,1,0
3453428,0,0.770748,85.275711,37.287338,1.000000,5,10.683108,8.085178,1.321320,0.987275,13.397720,6.902415,0,0,666.807983,1995.526978,1820.094238,0.000000,14,20317.841797,0.004197,6.890495,303.054779,982.378235,267.301361,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1866.536133,0,0,0,0,5000.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.000000,0.000000,0.000,0.0,0.000000,0.000000,1.000000,0,37.642460,296.397552,285.905243,1356.362915,63.777504,962.811096,0,0,0,0,0,0,1,0,0,0,2,0,3,0.333333,0.0,0.666667,0.0,0.0,1589.203979,0,0,230.784134,1,0,5000.0,0,0,1969.034546,0,0
3453429,0,0.738845,339.931122,76.036804,1.000000,5,23.667395,14.380991,1.645742,0.998738,27.694016,0.000000,1,1,651.821228,1996.703003,1832.733521,0.000000,14,20317.841797,0.016731,6.890495,287.138824,982.378235,272.490387,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1870.902832,0,0,0,0,5000.000000,

In [33]:
df_train_pool['stage1_l2'].value_counts()

stage1_l2
SFH                  1648916
MFH                  1470564
TH                    326675
farm_auxiliary        241815
retail                145200
utilities              97480
religious              74603
school                 58562
office                 55086
greenhouse             46423
kindergarten           32970
food_drink             32634
recreation             30928
accommodation          30622
storage                27466
emergency_service      25826
equestrian             16263
community              16117
manufacturing          12220
government_office      10583
hospital               10207
cultural               10089
higher_ed               9915
care_facility           9694
animal_keeping          8553
parking                 6891
maintenance             6562
clinic                  5929
bunker                  5356
train_station           4903
barracks                 785
airport                  207
research                  73
bus_station               51
Name

In [34]:
# Encode Targets (fit on full train_pool)


le_l1 = LabelEncoder()
le_l1.fit(df_train_pool['stage1_l1'])

le_l2 = LabelEncoder()
le_l2.fit(df_train_pool['stage1_l2'].dropna())

print("L1 classes:", list(le_l1.classes_))
print("L2 classes:", list(le_l2.classes_))
print("\nL2 by L1 parent (check your hierarchy makes sense):")
print(df_train_pool.groupby(
    ['stage1_l1', 'stage1_l2'], dropna=False
).size().to_string())

L1 classes: ['agricultural', 'civic', 'commercial', 'industrial', 'military', 'residential', 'transportation']
L2 classes: ['MFH', 'SFH', 'TH', 'accommodation', 'airport', 'animal_keeping', 'barracks', 'bunker', 'bus_station', 'care_facility', 'clinic', 'community', 'cultural', 'emergency_service', 'equestrian', 'farm_auxiliary', 'food_drink', 'government_office', 'greenhouse', 'higher_ed', 'hospital', 'kindergarten', 'maintenance', 'manufacturing', 'office', 'parking', 'recreation', 'religious', 'research', 'retail', 'school', 'storage', 'train_station', 'utilities']

L2 by L1 parent (check your hierarchy makes sense):
stage1_l1       stage1_l2        
agricultural    animal_keeping          8553
                equestrian             16263
                farm_auxiliary        241815
                greenhouse             46423
                NaN                      555
civic           care_facility           9694
                clinic                  5929
                communi

In [35]:
feature_cols

['address_tier',
 'compactness',
 'area',
 'perimeter',
 'convexity',
 'num_vertices',
 'mrr_long',
 'mrr_short',
 'elongation',
 'rectangularity',
 'diameter',
 'dist_nearest_real_building',
 'is_touching_real_building',
 'real_touching_count',
 'dist_to_residential',
 'dist_to_commercial',
 'dist_to_industrial',
 'dist_to_agricultural',
 'buildings_in_same_zone',
 'zone_total_area',
 'building_area_fraction',
 'zone_building_density',
 'neighbour_mean_area_200m',
 'neighbour_max_area_200m',
 'neighbour_std_area_200m',
 'poi_count_retail_25m',
 'poi_count_retail_50m',
 'poi_count_retail_100m',
 'poi_count_retail_250m',
 'poi_count_office_25m',
 'poi_count_office_50m',
 'poi_count_office_100m',
 'poi_count_office_250m',
 'poi_count_food_25m',
 'poi_count_food_50m',
 'poi_count_food_100m',
 'poi_count_food_250m',
 'poi_count_other_service_25m',
 'poi_count_other_service_50m',
 'poi_count_other_service_100m',
 'poi_count_other_service_250m',
 'poi_count_accommodation_25m',
 'poi_count_ac

In [36]:
# Train / Val / Test Split on df_train_pool
X = df_train_pool[feature_cols]
# Wrap the encoded labels in a Series with the original index
y_l1 = pd.Series(le_l1.transform(df_train_pool['stage1_l1']), index=df_train_pool.index)

# Stratify on L1 to preserve class balance across all three splits
# First Split (Extract 10% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_l1,
    test_size=0.10,
    stratify=y_l1,
    random_state=42,
)

# Second Split (Extract ~5% Val from remaining 90%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.05 / 0.90,   # ~5% of total
    stratify=y_train,
    random_state=42,
)

# Keep index alignment for L2 label extraction later
idx_train = X_train.index
idx_val = X_val.index
idx_test = X_test.index

print(f"Train : {len(X_train):,}  ({len(X_train)/len(X)*100:.1f}%)")
print(f"Val   : {len(X_val):,}    ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test  : {len(X_test):,}   ({len(X_test)/len(X)*100:.1f}%)")

for name, y_sp in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    print(f"\n{name} class balance:")
    for i, cls in enumerate(le_l1.classes_):
        n = (y_sp == i).sum()
        print(f"  {cls:<20}: {n:>8,}  ({n/len(y_sp)*100:.1f}%)")

Train : 7,772,053  (85.0%)
Val   : 457,180    (5.0%)
Test  : 914,360   (10.0%)

Train class balance:
  agricultural        :  266,567  (3.4%)
  civic               :  264,771  (3.4%)
  commercial          :  357,775  (4.6%)
  industrial          :  256,469  (3.3%)
  military            :    7,045  (0.1%)
  residential         : 6,600,470  (84.9%)
  transportation      :   18,956  (0.2%)

Val class balance:
  agricultural        :   15,681  (3.4%)
  civic               :   15,575  (3.4%)
  commercial          :   21,046  (4.6%)
  industrial          :   15,086  (3.3%)
  military            :      414  (0.1%)
  residential         :  388,263  (84.9%)
  transportation      :    1,115  (0.2%)

Test class balance:
  agricultural        :   31,361  (3.4%)
  civic               :   31,149  (3.4%)
  commercial          :   42,091  (4.6%)
  industrial          :   30,173  (3.3%)
  military            :      829  (0.1%)
  residential         :  776,527  (84.9%)
  transportation      :    2,230  

In [37]:
# Apply log transform to skewed features 
# Get the Series of skewness values (keep the numbers and the names)
skew_vals = gdf_bldg[feature_cols].skew().sort_values(ascending=False)

# Filter the Series where the NUMBER is > 5, THEN extract the names
SKEWED_COLS = skew_vals[skew_vals > 5].index

X_train_log = X_train.copy()
X_val_log   = X_val.copy()
X_test_log  = X_test.copy()
X_predict_log = X_predict.copy()

X_train_log[SKEWED_COLS] = np.log1p(X_train[SKEWED_COLS].astype(float))
X_val_log[SKEWED_COLS]   = np.log1p(X_val[SKEWED_COLS].astype(float))
X_test_log[SKEWED_COLS]  = np.log1p(X_test[SKEWED_COLS].astype(float))
X_predict_log[SKEWED_COLS] = np.log1p(X_predict[SKEWED_COLS].astype(float))


# Fit Scaler on X_train ONLY, transform everything else
# Only scaling continuous features — binary flags and int counts stay raw

SCALE_COLS = [
    'area', 'compactness', 'convexity', 'num_vertices',
    'mrr_long', 'mrr_short', 'elongation', 'rectangularity', 
    'perimeter', 'diameter', 'dist_nearest_building', 
    'dist_to_residential', 'dist_to_commercial', 'dist_to_industrial',
    'dist_nearest_education', 'dist_nearest_transport',
    'dist_nearest_residential_road', 'dist_nearest_secondary_road',
    'dist_nearest_rural_track', 'dist_nearest_major_road',
    'dist_nearest_service_road', 'dist_nearest_pedestrian_zone', 'touching_building_count'
]

poi_count_cols = [c for c in feature_cols if c.startswith('poi_count_')]
SCALE_COLS.extend(poi_count_cols)

SCALE_COLS = [c for c in SCALE_COLS if c in feature_cols]


scaler = StandardScaler()
scaler.fit(X_train_log[SCALE_COLS])   # fit on train only

def apply_scale(X, scaler, cols):
    X_scaled = X.copy()
    X_scaled[cols] = scaler.transform(X[cols]).astype(np.float32)
    return X_scaled

X_train_scaled   = apply_scale(X_train_log,            scaler, SCALE_COLS)
X_val_scaled     = apply_scale(X_val_log,              scaler, SCALE_COLS)
X_test_scaled    = apply_scale(X_test_log,             scaler, SCALE_COLS)
X_predict_scaled = apply_scale(X_predict_log[feature_cols],   scaler, SCALE_COLS)

print(f"Scaler fitted on {len(X_train_log):,} training rows")
print(f"Scaled {len(SCALE_COLS)} continuous features: {SCALE_COLS}")
print("Raw X_train, X_val, X_test preserved for tree models")

Scaler fitted on 7,772,053 training rows
Scaled 54 continuous features: ['area', 'compactness', 'convexity', 'num_vertices', 'mrr_long', 'mrr_short', 'elongation', 'rectangularity', 'perimeter', 'diameter', 'dist_to_residential', 'dist_to_commercial', 'dist_to_industrial', 'dist_nearest_education', 'dist_nearest_transport', 'dist_nearest_residential_road', 'dist_nearest_secondary_road', 'dist_nearest_rural_track', 'dist_nearest_major_road', 'dist_nearest_service_road', 'dist_nearest_pedestrian_zone', 'poi_count_retail_25m', 'poi_count_retail_50m', 'poi_count_retail_100m', 'poi_count_retail_250m', 'poi_count_office_25m', 'poi_count_office_50m', 'poi_count_office_100m', 'poi_count_office_250m', 'poi_count_food_25m', 'poi_count_food_50m', 'poi_count_food_100m', 'poi_count_food_250m', 'poi_count_other_service_25m', 'poi_count_other_service_50m', 'poi_count_other_service_100m', 'poi_count_other_service_250m', 'poi_count_accommodation_25m', 'poi_count_accommodation_50m', 'poi_count_accommoda

In [38]:
X_train_scaled.head()

,address_tier,compactness,area,perimeter,convexity,num_vertices,mrr_long,mrr_short,elongation,rectangularity,diameter,dist_nearest_real_building,is_touching_real_building,real_touching_count,dist_to_residential,dist_to_commercial,dist_to_industrial,dist_to_agricultural,buildings_in_same_zone,zone_total_area,building_area_fraction,zone_building_density,neighbour_mean_area_200m,neighbour_max_area_200m,neighbour_std_area_200m,poi_count_retail_25m,poi_count_retail_50m,poi_count_retail_100m,poi_count_retail_250m,poi_count_office_25m,poi_count_office_50m,poi_count_office_100m,poi_count_office_250m,poi_count_food_25m,poi_count_food_50m,poi_count_food_100m,poi_count_food_250m,poi_count_other_service_25m,poi_count_other_service_50m,poi_count_other_service_100m,poi_count_other_service_250m,poi_count_accommodation_25m,poi_count_accommodation_50m,poi_count_accommodation_100m,poi_count_accommodation_250m,poi_count_healthcare_25m,poi_count_healthcare_50m,poi_count_healthcare_100m,poi_count_healthcare_250m,poi_count_education_25m,poi_count_education_50m,poi_count_education_100m,poi_count_education_250m,dist_nearest_education,poi_count_civic_25m,poi_count_civic_50m,poi_count_civic_100m,poi_count_civic_250m,dist_nearest_transport,food_share_100m,retail_share_100m,office_share_100m,accom_share_100m,food_vs_retail_100m,office_vs_retail_100m,comm_diversity_100m,food_share_250m,retail_share_250m,office_share_250m,accom_share_250m,food_vs_retail_250m,office_vs_retail_250m,comm_diversity_250m,poi_count_inside,dist_nearest_residential_road,dist_nearest_secondary_road,dist_nearest_rural_track,dist_nearest_major_road,dist_nearest_service_road,dist_nearest_pedestrian_zone,near_major_road_700m,near_secondary_road_115m,near_residential_road_25m,near_service_road_35m,near_rural_track_250m,near_pedestrian_zone_25m,road_count_residential_road_200m,road_count_secondary_road_200m,road_count_rural_track_200m,road_count_major_road_200m,road_count_service_road_200m,road_count_pedestrian_zone_200m,road_count_total_200m,pct_residential_roads_200m,pct_rural_roads_200m,pct_service_roads_200m,pct_major_roads_200m,pct_secondary_roads_200m,dist_nearest_major_waterway,near_major_waterway_500m,count_major_waterway_200m,dist_nearest_minor_waterway,near_minor_waterway_300m,count_minor_waterway_200m,dist_nearest_light_rail,near_light_rail_200m,count_light_rail_200m,dist_nearest_heavy_rail,near_heavy_rail_500m,count_heavy_rail_200m
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
127973602,2,0.000388,-0.370610,-0.348682,0.537653,-0.777369,-0.103720,-0.780287,0.899815,0.734385,-0.309100,0.0,1,1,1.894587,-0.774986,-0.509468,480.294403,0,0.000000,0.000000,0.000000,4.801161,8.245655,5.502041,-0.156545,-0.234382,-0.365989,0.810588,-0.0721,-0.130356,-0.228999,1.049092,-0.14563,-0.226964,-0.349866,1.014010,-0.106361,-0.172405,-0.268508,1.220219,-0.010138,-0.017771,-0.032625,-0.068371,-0.083214,-0.149941,-0.264078,2.232149,-0.087423,-0.153605,-0.299089,-0.536470,-0.429624,-0.134721,-0.232523,-0.410276,1.498557,-1.452129,0.000000,0.000000,0.0,0.0,0.00000,0.0,1.000000,0.333296,0.333296,0.111099,0.0,0.499917,0.249938,0.765484,-0.168447,0.261595,-0.742294,-0.007483,1.119654,0.396518,-1.035492,0,1,0,0,0,0.0,22,12,0,0.0,15,0.0,49,0.448980,0.000000,0.306122,0.0,0.244898,2845.445068,0,0.0,53.982670,1,0.693147,5000.000000,0.0,0.0,1073.651367,0,0.000000
203755731,0,0.121802,0.884752,0.758222,0.465009,0.028868,0.914703,0.453242,0.704054,0.640641,0.788177,0.0,1,2,-0.382096,-0.788548,-1.013146,530.714600,0,0.000000,0.000000,0.000000,5.434873,8.354220,6.055812,-0.156545,-0.234382,2.444228,1.247315,-0.0721,-0.130356,-0.228999,1.049092,-0.14563,2.674682,4.423511,2.513827,-0.106361,-0.172405,1.962066,2.008308,-0.010138,-0.017771,-0.032625,-0.068371,9.914607,5.076080,2.390942,1.851243,-0.087423,-0.153605,-0.299089,0.267128,-0.582901,6.489005,3.343269,2.494483,1.498557,-1.317070,0.636306,0.272702,0.0,0.0,0.69993,0.0,0.520748,0.565193,0.217

In [40]:
# Build Hierarchical Label Arrays

def build_hierarchical_splits(df_pool, idx_train, idx_val, idx_test,
                               X_train, X_val, X_test,
                               le_l1, le_l2):
    splits = {}

    # ── FLAT L1: all L1 classes in one model ─────────────────────────
    splits['flat_l1'] = {
        'train': (X_train, le_l1.transform(df_pool.loc[idx_train, 'stage1_l1'])),
        'val'  : (X_val,   le_l1.transform(df_pool.loc[idx_val,   'stage1_l1'])),
        'test' : (X_test,  le_l1.transform(df_pool.loc[idx_test,  'stage1_l1'])),
        'classes': le_l1.classes_,
    }

    # ── FLAT L2: all subtypes in one model ───────────────────────────
    for split_name, idx_sp, X_sp in [
        ('train', idx_train, X_train),
        ('val',   idx_val,   X_val),
        ('test',  idx_test,  X_test),
    ]:
        sub      = df_pool.loc[idx_sp]
        has_l2   = sub['stage1_l2'].notna()
        X_sub    = X_sp[has_l2.values]
        y_sub    = le_l2.transform(sub.loc[has_l2, 'stage1_l2'])
        splits.setdefault('flat_l2', {})[split_name] = (X_sub, y_sub)
    splits['flat_l2']['classes'] = le_l2.classes_

    # ── HIERARCHICAL ─────────────────────────────────────────────────
    RES_CLASSES  = [c for c in le_l1.classes_ if 'residential' in c.lower()]
    IND_CLASSES  = [c for c in le_l1.classes_ if 'industrial'  in c.lower()]
    COMM_CLASSES = [c for c in le_l1.classes_ if 'commercial'  in c.lower()]

    # AFTER — restrict to train indices only
    train_pool = df_pool.loc[df_pool.index.isin(idx_train)]

    le_res = LabelEncoder()
    le_res.fit(train_pool.loc[train_pool['stage1_l1'].isin(RES_CLASSES) & train_pool['stage1_l2'].notna(), 'stage1_l2'])

    le_ind = LabelEncoder()
    le_ind.fit(train_pool.loc[train_pool['stage1_l1'].isin(IND_CLASSES) & train_pool['stage1_l2'].notna(), 'stage1_l2'])

    le_comm = LabelEncoder()
    le_comm.fit(train_pool.loc[train_pool['stage1_l1'].isin(COMM_CLASSES) & train_pool['stage1_l2'].notna(), 'stage1_l2'])
    
    for key in ['l1a_res_subtypes', 'l2a_ind_subtypes', 'l3a_comm_subtypes']:
        splits[key] = {}

    for split_name, idx_sp, X_sp in [
        ('train', idx_train, X_train),
        ('val',   idx_val,   X_val),
        ('test',  idx_test,  X_test),
    ]:
        sub   = df_pool.loc[idx_sp]

        # L0: residential vs non-residential
        is_res = np.isin(sub['stage1_l1'].values, RES_CLASSES)
        
        y_res_binary = (~is_res).astype(np.int8) 
        splits.setdefault('l0_res_binary', {})[split_name] = (X_sp, y_res_binary)

        
        # L1a: residential subtypes
        res_mask = is_res
        sub_res  = sub[res_mask]
        has_l2   = sub_res['stage1_l2'].notna()
        if has_l2.any():
            splits.setdefault('l1a_res_subtypes', {})[split_name] = (
                X_sp[res_mask][has_l2.values],
                le_res.transform(sub_res.loc[has_l2, 'stage1_l2'])
            )

        # L1b: industrial vs non-industrial (within non-res)
        nonres_mask = ~is_res
        sub_nonres  = sub[nonres_mask]
        y_ind = np.isin(sub_nonres['stage1_l1'].values, IND_CLASSES).astype(np.int8)
        splits.setdefault('l1b_ind_binary', {})[split_name] = (
            X_sp[nonres_mask], y_ind
        )

        # L2a: industrial subtypes
        ind_mask = nonres_mask.copy()
        ind_mask[nonres_mask] = y_ind == 1
        assert ind_mask.sum() == (y_ind == 1).sum(), \
        f"ind_mask mismatch: {ind_mask.sum()} vs {(y_ind==1).sum()}"
        sub_ind  = sub[ind_mask]
        has_l2   = sub_ind['stage1_l2'].notna()
        if has_l2.any():
            splits.setdefault('l2a_ind_subtypes', {})[split_name] = (
                X_sp[ind_mask][has_l2.values],
                le_ind.transform(sub_ind.loc[has_l2, 'stage1_l2']) 
            )

        # L2b: commercial vs other (non-res, non-ind)
        comm_other_mask = nonres_mask.copy()
        comm_other_mask[nonres_mask] = y_ind == 0
        assert comm_other_mask.sum() == (y_ind == 0).sum(), \
        f"comm_other_mask mismatch"
        sub_co   = sub[comm_other_mask]
        y_comm   = np.isin(sub_co['stage1_l1'].values, COMM_CLASSES).astype(np.int8)
        splits.setdefault('l2b_comm_binary', {})[split_name] = (
            X_sp[comm_other_mask], y_comm
        )

        # L3a: commercial subtypes
        comm_mask = comm_other_mask.copy()
        comm_mask[comm_other_mask] = y_comm == 1
        sub_comm = sub[comm_mask]
        has_l2   = sub_comm['stage1_l2'].notna()
        if has_l2.any():
            splits.setdefault('l3a_comm_subtypes', {})[split_name] = (
                X_sp[comm_mask][has_l2.values],
                le_comm.transform(sub_comm.loc[has_l2, 'stage1_l2'])
            )

    # Add class labels for reference
    splits['l0_res_binary']['classes']  = np.array(['residential', 'non_residential'])
    splits['l1b_ind_binary']['classes'] = np.array(['non_industrial', 'industrial'])
    splits['l2b_comm_binary']['classes']= np.array(['other', 'commercial'])

    splits['l1a_res_subtypes']['classes'] = le_res.classes_
    splits['l2a_ind_subtypes']['classes'] = le_ind.classes_
    splits['l3a_comm_subtypes']['classes']= le_comm.classes_

    splits['encoders'] = {
        'res': le_res,
        'ind': le_ind,
        'comm': le_comm
    }

    print("\nVal set binary balance check:")
    for key in ['l0_res_binary', 'l1b_ind_binary', 'l2b_comm_binary']:
        _, y_v = splits[key]['val']
        counts = np.bincount(y_v.astype(int))
        labels = splits[key]['classes']
        for lbl, cnt in zip(labels, counts):
            print(f"  {key} | {lbl}: {cnt:,} ({cnt/len(y_v)*100:.1f}%)")

    return splits


splits = build_hierarchical_splits(
    df_train_pool, idx_train, idx_val, idx_test,
    X_train, X_val, X_test,
    le_l1, le_l2
)

print("Splits built:")
for name, sp in splits.items():
    if name == 'encoders':
        continue
    Xtr, ytr = sp['train']
    Xva, yva = sp['val']
    Xte, yte = sp['test']
    print(f"  {name:<25}: train={len(Xtr):,}  val={len(Xva):,}  test={len(Xte):,}  classes={len(np.unique(ytr))}")


Val set binary balance check:
  l0_res_binary | residential: 388,263 (84.9%)
  l0_res_binary | non_residential: 68,917 (15.1%)
  l1b_ind_binary | non_industrial: 53,831 (78.1%)
  l1b_ind_binary | industrial: 15,086 (21.9%)
  l2b_comm_binary | other: 32,785 (60.9%)
  l2b_comm_binary | commercial: 21,046 (39.1%)
Splits built:
  flat_l1                  : train=7,772,053  val=457,180  test=914,360  classes=7
  flat_l2                  : train=3,808,232  val=224,126  test=447,810  classes=34
  l1a_res_subtypes         : train=2,929,004  val=172,402  test=344,749  classes=3
  l2a_ind_subtypes         : train=116,682  val=6,931  test=13,553  classes=3
  l3a_comm_subtypes        : train=224,171  val=13,166  test=26,205  classes=4
  l0_res_binary            : train=7,772,053  val=457,180  test=914,360  classes=2
  l1b_ind_binary           : train=1,171,583  val=68,917  test=137,833  classes=2
  l2b_comm_binary          : train=915,114  val=53,831  test=107,660  classes=2


In [41]:
splits_scaled = build_hierarchical_splits(
    df_train_pool, idx_train, idx_val, idx_test,
    X_train_scaled, X_val_scaled, X_test_scaled,
    le_l1, le_l2
)

print("Scaled splits built:")
for name, sp in splits_scaled.items():
    if name == 'encoders':
        continue
    Xtr, ytr = sp['train']
    Xva, yva = sp['val']
    Xte, yte = sp['test']
    print(f"  {name:<25}: train={len(Xtr):,}  val={len(Xva):,}  test={len(Xte):,}  classes={len(np.unique(ytr))}")

# Sanity check — y arrays must be identical between raw and scaled
for name in splits:
    if name == 'encoders':
        continue
    assert np.array_equal(splits[name]['train'][1], splits_scaled[name]['train'][1]), \
        f"Label mismatch in {name}!"
print("\n✅ Label arrays identical between raw and scaled splits.")


Val set binary balance check:
  l0_res_binary | residential: 388,263 (84.9%)
  l0_res_binary | non_residential: 68,917 (15.1%)
  l1b_ind_binary | non_industrial: 53,831 (78.1%)
  l1b_ind_binary | industrial: 15,086 (21.9%)
  l2b_comm_binary | other: 32,785 (60.9%)
  l2b_comm_binary | commercial: 21,046 (39.1%)
Scaled splits built:
  flat_l1                  : train=7,772,053  val=457,180  test=914,360  classes=7
  flat_l2                  : train=3,808,232  val=224,126  test=447,810  classes=34
  l1a_res_subtypes         : train=2,929,004  val=172,402  test=344,749  classes=3
  l2a_ind_subtypes         : train=116,682  val=6,931  test=13,553  classes=3
  l3a_comm_subtypes        : train=224,171  val=13,166  test=26,205  classes=4
  l0_res_binary            : train=7,772,053  val=457,180  test=914,360  classes=2
  l1b_ind_binary           : train=1,171,583  val=68,917  test=137,833  classes=2
  l2b_comm_binary          : train=915,114  val=53,831  test=107,660  classes=2

✅ Label array

In [42]:
# Save 
# Raw splits (tree models) & Hierarchical Data
for name, sp in splits.items():
    if name == 'encoders':
        continue
    for part in ['train', 'val', 'test']:
        if part in sp:
            joblib.dump(sp[part], os.path.join(ML_DIR, f'{name}_{part}_raw.pkl'))
    joblib.dump(sp['classes'], os.path.join(ML_DIR, f'{name}_classes.pkl'))

# Scaled splits (NN/GNN)
for name, sp in splits_scaled.items():
    if name == 'encoders':
        continue
    for part in ['train', 'val', 'test']:
        if part in sp:
            joblib.dump(sp[part], os.path.join(ML_DIR, f'{name}_{part}_scaled.pkl'))


# Encoders, scaler, feature list
joblib.dump(le_l1,        os.path.join(ML_DIR, 'label_encoder_l1.pkl'))
joblib.dump(le_l2,        os.path.join(ML_DIR, 'label_encoder_l2.pkl'))
joblib.dump(splits['encoders']['res'],  os.path.join(ML_DIR, 'label_encoder_res.pkl'))
joblib.dump(splits['encoders']['ind'],  os.path.join(ML_DIR, 'label_encoder_ind.pkl'))
joblib.dump(splits['encoders']['comm'], os.path.join(ML_DIR, 'label_encoder_comm.pkl'))
joblib.dump(scaler,       os.path.join(ML_DIR, 'scaler.pkl'))
joblib.dump(SCALE_COLS,   os.path.join(ML_DIR, 'scale_cols.pkl'))
joblib.dump(feature_cols, os.path.join(ML_DIR, 'candidate_features.pkl'))
joblib.dump(df_train_pool, os.path.join(ML_DIR, 'df_train_pool.pkl'))

# Predict set (true inference — no labels)
joblib.dump(X_predict[feature_cols],        os.path.join(ML_DIR, 'predict_raw.pkl'))
joblib.dump(X_predict_scaled,        os.path.join(ML_DIR, 'predict_scaled.pkl'))

# Stage2 set (set aside)
joblib.dump(df_stage2, os.path.join(ML_DIR, 'stage2_raw.pkl'))

# save excluded st
joblib.dump(df_excluded, os.path.join(ML_DIR, 'excluded_full.pkl'))

print(f"Saved to {ML_DIR}/")
for f in sorted(os.listdir(ML_DIR)):
    size = os.path.getsize(os.path.join(ML_DIR, f)) / 1e6
    print(f"  {f:<50} {size:.1f} MB")

Saved to /fast/home/o-olajuyigbe/osm_project/data/ml_ready/
  candidate_features.pkl                             0.0 MB
  candidate_features_height.pkl                      0.0 MB
  df_train_pool.pkl                                  3671.0 MB
  df_train_pool_height.pkl                           3826.4 MB
  excluded_full.pkl                                  1656.6 MB
  excluded_full_height.pkl                           1726.8 MB
  feature_cols.pkl                                   0.0 MB
  flat_l1_classes.pkl                                0.0 MB
  flat_l1_classes_height.pkl                         0.0 MB
  flat_l1_test_raw.pkl                               369.4 MB
  flat_l1_test_raw_height.pkl                        384.9 MB
  flat_l1_test_scaled.pkl                            457.2 MB
  flat_l1_test_scaled_height.pkl                     480.0 MB
  flat_l1_train_raw.pkl                              3139.8 MB
  flat_l1_train_raw_height.pkl                       3271.9 MB
  flat_l1_trai

In [43]:
# ── Per-level prediction sets for stage2 buildings ──────────────────────────
# These are rule-classified buildings. They have stage2_l1 but most lack stage2_l2.
# Each subtype model needs its own filtered slice.

# defined above but reiterated here for clarity:
df_stage2 = gdf_bldg[
    gdf_bldg['stage1_l1'].isna() &
    gdf_bldg['stage2_l1'].notna()
].copy()

# Identify which stage2 buildings need subtype prediction
# (they have a type label but no subtype label)

RES_CLASSES  = [c for c in le_l1.classes_ if 'residential' in c.lower()]
IND_CLASSES  = [c for c in le_l1.classes_ if 'industrial'  in c.lower()]
COMM_CLASSES = [c for c in le_l1.classes_ if 'commercial'  in c.lower()]

# L1a: stage2 residentials without a residential subtype
stage2_res_predict_mask = (
    df_stage2['stage2_l1'].isin(RES_CLASSES) &
    df_stage2['stage2_l2'].isna()
)
X_stage2_res_predict = df_stage2.loc[stage2_res_predict_mask, feature_cols]

# L2a: stage2 industrials without an industrial subtype
stage2_ind_predict_mask = (
    df_stage2['stage2_l1'].isin(IND_CLASSES) &
    df_stage2['stage2_l2'].isna()
)
X_stage2_ind_predict = df_stage2.loc[stage2_ind_predict_mask, feature_cols]

# L3a: stage2 commercials without a commercial subtype
stage2_comm_predict_mask = (
    df_stage2['stage2_l1'].isin(COMM_CLASSES) &
    df_stage2['stage2_l2'].isna()
)
X_stage2_comm_predict = df_stage2.loc[stage2_comm_predict_mask, feature_cols]

# stage2 buildings that need full cascade (no l1 at all from stage2 either)
stage2_needs_l0_mask = df_stage2['stage2_l1'].isna()  
print(f"Stage2 needs l0 prediction (should be 0): {stage2_needs_l0_mask.sum():,}")

print(f"Stage2 res needing subtype prediction : {stage2_res_predict_mask.sum():,}")
print(f"Stage2 ind needing subtype prediction : {stage2_ind_predict_mask.sum():,}")
print(f"Stage2 comm needing subtype prediction: {stage2_comm_predict_mask.sum():,}")

# Save
joblib.dump(df_stage2,                  os.path.join(ML_DIR, 'stage2_full.pkl'))
joblib.dump(X_stage2_res_predict,       os.path.join(ML_DIR, 'stage2_l1a_res_predict.pkl'))
joblib.dump(X_stage2_ind_predict,       os.path.join(ML_DIR, 'stage2_l2a_ind_predict.pkl'))
joblib.dump(X_stage2_comm_predict,      os.path.join(ML_DIR, 'stage2_l3a_comm_predict.pkl'))

Stage2 needs l0 prediction (should be 0): 0
Stage2 res needing subtype prediction : 1,433,862
Stage2 ind needing subtype prediction : 44,813
Stage2 comm needing subtype prediction: 24,931


['/fast/home/o-olajuyigbe/osm_project/data/ml_ready/stage2_l3a_comm_predict.pkl']

In [44]:
# ── Training pool buildings without subtype labels (need subtype prediction) ──
# These were in the training pool for the L0/L1b/L2b binary models,
# but couldn't contribute to subtype training. Post-training they need subtypes assigned.

trainpool_res_no_l2 = df_train_pool[
    df_train_pool['stage1_l1'].isin(RES_CLASSES) &
    df_train_pool['stage1_l2'].isna()
]
trainpool_ind_no_l2 = df_train_pool[
    df_train_pool['stage1_l1'].isin(IND_CLASSES) &
    df_train_pool['stage1_l2'].isna()
]
trainpool_comm_no_l2 = df_train_pool[
    df_train_pool['stage1_l1'].isin(COMM_CLASSES) &
    df_train_pool['stage1_l2'].isna()
]

print(f"Trainpool res without subtype : {len(trainpool_res_no_l2):,}")
print(f"Trainpool ind without subtype : {len(trainpool_ind_no_l2):,}")
print(f"Trainpool comm without subtype: {len(trainpool_comm_no_l2):,}")

joblib.dump(trainpool_res_no_l2[feature_cols],  os.path.join(ML_DIR, 'trainpool_l1a_res_predict.pkl'))
joblib.dump(trainpool_ind_no_l2[feature_cols],  os.path.join(ML_DIR, 'trainpool_l2a_ind_predict.pkl'))
joblib.dump(trainpool_comm_no_l2[feature_cols], os.path.join(ML_DIR, 'trainpool_l3a_comm_predict.pkl'))

Trainpool res without subtype : 4,319,105
Trainpool ind without subtype : 164,562
Trainpool comm without subtype: 157,370


['/fast/home/o-olajuyigbe/osm_project/data/ml_ready/trainpool_l3a_comm_predict.pkl']